In [ ]:
import pandas as pd

df = pd.read_csv('research/phase0/data/MERGED_XAUUSDm_M1_2020_01_01_to_2026_05_07.csv')
df['Time'] = pd.to_datetime(df['Time'], format='%Y.%m.%d %H:%M')
df = df.set_index('Time').sort_index()

print(f"Total rows: {len(df):,}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Duplicates: {df.index.duplicated().sum()}")

# Find gaps larger than 4 hours (filters out normal 1-min gaps)
time_diffs = df.index.to_series().diff()
gaps = time_diffs[time_diffs > pd.Timedelta(hours=4)]

# Categorize gaps
weekend_gaps = gaps[(gaps >= pd.Timedelta(hours=40)) & (gaps <= pd.Timedelta(hours=56))]
suspicious_gaps = gaps[(gaps > pd.Timedelta(hours=4)) & (gaps < pd.Timedelta(hours=40))]
huge_gaps = gaps[gaps > pd.Timedelta(hours=56)]

print(f"\nWeekend gaps (40-56 hours, normal):  {len(weekend_gaps)}")
print(f"Suspicious gaps (4-40 hours):        {len(suspicious_gaps)}")
print(f"Huge gaps (>56 hours, missing data): {len(huge_gaps)}")

# Show only the problematic gaps
if len(suspicious_gaps) > 0:
    print("\n=== SUSPICIOUS GAPS (4-40 hours) ===")
    print("These might indicate missing trading hours or merge issues:")
    for ts, gap in suspicious_gaps.items():
        hours = gap.total_seconds() / 3600
        print(f"  {ts}  ->  gap of {hours:.1f} hours")

if len(huge_gaps) > 0:
    print("\n=== HUGE GAPS (>2.5 days) ===")
    print("These indicate missing data chunks or extended market closures:")
    for ts, gap in huge_gaps.items():
        days = gap.total_seconds() / 86400
        print(f"  {ts}  ->  gap of {days:.1f} days")

# Rows per year — quick visual check
print("\n=== ROWS PER YEAR ===")
yearly = df.groupby(df.index.year).size()
for year, count in yearly.items():
    bar = '#' * int(count / 10000)
    print(f"  {year}: {count:>7,}  {bar}")

# Rows per month for last 24 months
print("\n=== ROWS PER MONTH (last 24 months) ===")
monthly = df.groupby([df.index.year, df.index.month]).size().tail(24)
for (year, month), count in monthly.items():
    bar = '#' * int(count / 1000)
    print(f"  {year}-{month:02d}: {count:>6,}  {bar}")